# Strands 멀티 에이전트 시스템과 AgentCore 메모리 도구 (단기 메모리)

## 소개

이 노트북은 AWS AgentCore 메모리와 Strands 프레임워크를 사용하여 **공유 메모리가 있는 멀티 에이전트 시스템**을 구현하는 방법을 보여줍니다. 이전 예제에서는 단일 에이전트 메모리에 초점을 맞추었지만, 이 노트북에서는 여러 전문화된 에이전트가 공통 메모리 저장소에 접근하면서 함께 작업하는 방법을 탐구합니다.

## 튜토리얼 세부 정보

| 항목                | 세부 내용                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화형                                                                      |
| 에이전트 활용 사례  | 여행 계획 어시스턴트                                                             |
| 에이전트 프레임워크 | Strands Agents                                                                   |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                   |
| 튜토리얼 구성 요소  | AgentCore 단기 메모리, Strands Agents, 도구를 통한 메모리 조회                   |
| 예제 난이도         | 초급                                                                             |


학습 내용:

- 여러 에이전트가 접근할 수 있는 공유 메모리 리소스 설정 방법
- 자체 메모리 접근 권한을 가진 전문화된 에이전트를 도구로 생성
- 전문화된 에이전트에 위임하는 코디네이터 에이전트 구현
- 여러 에이전트 상호작용에 걸친 대화 컨텍스트 유지

### 시나리오 컨텍스트

이 예제에서는 다음과 같은 **여행 계획 시스템**을 만들겠습니다:
1. 항공 여행 전문 항공편 예약 어시스턴트
2. 숙박 전문 호텔 예약 어시스턴트
3. 이 전문 에이전트들에게 위임하는 여행 코디네이터

이 접근 방식은 복잡한 도메인을 메모리 저장소를 공유하는 전문화된 에이전트로 분해하는 방법을 보여줍니다.

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## 사전 요구사항
- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리에 대한 적절한 권한이 있는 AWS IAM 역할
- Amazon Bedrock 모델 접근 권한

환경 설정과 공유 메모리 리소스 생성부터 시작하겠습니다!

## 1단계: 환경 설정
이 노트북을 실행하는 데 필요한 모든 라이브러리를 임포트하고 클라이언트를 정의하는 것부터 시작하겠습니다.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
from datetime import datetime
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent

Amazon Bedrock 모델 및 AgentCore에 대한 적절한 권한이 있는 리전과 역할을 정의합니다.

In [ ]:
import os
region = os.getenv('AWS_REGION', 'us-west-2')
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("agentcore-memory")

## 2단계: 공유 메모리 생성
이 섹션에서는 전문화된 에이전트들이 공유할 메모리 리소스를 생성합니다.

In [ ]:
from bedrock_agentcore.memory import MemoryClient

In [ ]:
client = MemoryClient(region_name=region)
memory_name = "TravelAgent_STM_%s" % datetime.now().strftime("%Y%m%d%H%M%S")
memory_id = None


In [ ]:
from botocore.exceptions import ClientError

try:
    print("메모리 생성 중...")
    memory_name = memory_name

    # 메모리 리소스 생성
    memory = client.create_memory_and_wait(
        name=memory_name,                       # 이 메모리 저장소의 고유 이름
        description="Travel Agent STM",         # 사람이 읽을 수 있는 설명
        strategies=[],                          # 단기 메모리를 위한 특별한 메모리 전략 없음
        event_expiry_days=7,                    # 7일 후 메모리 만료
        max_wait=300,                           # 메모리 생성 대기 최대 시간 (5분)
        poll_interval=10                        # 10초마다 상태 확인
    )

    # 메모리 ID 추출 및 출력
    memory_id = memory['id']
    print(f"메모리 생성 성공, ID: {memory_id}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ValidationException' and "already exists" in str(e):
        # 메모리가 이미 존재하는 경우, ID를 조회
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"메모리가 이미 존재합니다. 기존 메모리 ID 사용: {memory_id}")
except Exception as e:
    # 메모리 생성 중 오류 처리
    print(f"오류: {e}")
    import traceback
    traceback.print_exc()

    # 오류 시 정리 - 부분적으로 생성된 메모리 삭제
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"메모리 정리 완료: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"메모리 정리 실패: {cleanup_error}")

### 멀티 에이전트 시스템을 위한 공유 메모리 이해

우리가 생성한 메모리 리소스는 여행 계획 시스템의 공유 지식 기반 역할을 합니다. 모든 에이전트가 이 공통 메모리 저장소에서 읽고 쓰며, 다음을 가능하게 합니다:

1. **지식 일관성**: 모든 에이전트가 동일한 정보로 작업
2. **컨텍스트 보존**: 에이전트 전환 시에도 대화 기록 유지
3. **전문화된 접근**: 각 에이전트는 자체 actor_id를 갖지만 session_id를 공유

이 접근 방식을 통해 전문화된 에이전트들이 전체 대화 컨텍스트의 이점을 누리면서도 자신의 도메인에 집중할 수 있습니다.

## 3단계: 메모리 훅 프로바이더 생성

이 단계에서는 메모리 작업을 자동화하는 커스텀 `MemoryHookProvider` 클래스를 정의합니다. 훅은 에이전트 실행 생명주기의 특정 시점에 실행되는 특수 함수입니다. 여기서 만드는 메모리 훅은 두 가지 주요 기능을 수행합니다:

1. **메모리 조회**: 사용자가 메시지를 보낼 때 자동으로 관련 과거 대화를 가져옴
2. **메모리 저장**: 에이전트가 응답한 후 새로운 대화를 저장

이를 통해 수동 관리 없이 원활한 메모리 경험을 제공합니다.

In [ ]:
class ShortTermMemoryHook(HookProvider):
    def __init__(self, memory_client: MemoryClient, memory_id: str):
        self.memory_client = memory_client
        self.memory_id = memory_id
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """에이전트 시작 시 최근 대화 기록을 로드"""
        try:
            # 에이전트 상태에서 세션 정보 가져오기
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")
            
            if not actor_id or not session_id:
                logger.warning("에이전트 상태에 actor_id 또는 session_id가 없습니다")
                return
            
            # 최근 5개의 대화 턴 가져오기
            recent_turns = self.memory_client.get_last_k_turns(
                memory_id=self.memory_id,
                actor_id=actor_id,
                session_id=session_id,
                k=5,
                branch_name="main"
            )
            
            if recent_turns:
                # 대화 기록을 컨텍스트로 포맷팅
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message['role'].lower()
                        content = message['content']['text']
                        context_messages.append(f"{role.title()}: {content}")
                
                context = "\n".join(context_messages)
                logger.info(f"메모리에서 가져온 컨텍스트: {context}")
                
                # 에이전트의 시스템 프롬프트에 컨텍스트 추가
                event.agent.system_prompt += f"\n\nRecent conversation history:\n{context}\n\nContinue the conversation naturally based on this context."
                
                logger.info(f"최근 대화 턴 {len(recent_turns)}개 로드 완료")
            else:
                logger.info("이전 대화 기록이 없습니다")
                
        except Exception as e:
            logger.error(f"대화 기록 로드 실패: {e}")
    
    def on_message_added(self, event: MessageAddedEvent):
        """메모리에 대화 턴 저장"""
        messages = event.agent.messages
        try:
            # 에이전트 상태에서 세션 정보 가져오기
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")
            
            if not actor_id or not session_id:
                logger.warning("에이전트 상태에 actor_id 또는 session_id가 없습니다")
                return
            
            self.memory_client.create_event(
                memory_id=self.memory_id,
                actor_id=actor_id,
                session_id=session_id,
                messages=[(messages[-1]["content"][0]["text"], messages[-1]["role"])]
            )
            
        except Exception as e:
            logger.error(f"메시지 저장 실패: {e}")
    
    def register_hooks(self, registry: HookRegistry) -> None:
        # 메모리 훅 등록
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

## 4단계: Strands 에이전트로 멀티 에이전트 아키텍처 생성
이 섹션에서는 항공편과 호텔 예약을 위한 전문화된 에이전트를 생성하고, 모두 메모리 리소스에 대한 공유 접근 권한을 갖도록 합니다.

In [ ]:
# 필요한 구성 요소 임포트
from strands import Agent, tool

In [ ]:
# 각 전문 에이전트에 고유 액터 ID를 생성하되 세션 ID는 공유
flight_actor_id = f"flight-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
hotel_actor_id = f"hotel-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"travel-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"
flight_namespace = f"travel/{flight_actor_id}/preferences/"
hotel_namespace = f"travel/{hotel_actor_id}/preferences/"

### 메모리 접근이 가능한 전문화된 에이전트 생성

다음으로, 전문화된 에이전트들의 시스템 프롬프트를 정의하겠습니다. 각 프롬프트에는 에이전트가 파싱할 수 있는 형식의 메모리 파라미터가 포함됩니다:

In [ ]:
# 시스템 프롬프트: 호텔 예약 전문가 - 호텔 검색, 예약, 숙박 및 편의시설 안내
HOTEL_BOOKING_PROMPT = f"""You are a hotel booking assistant. Help customers find hotels, make reservations, and answer questions about accommodations and amenities. 
Provide clear information about availability, pricing, and booking procedures in a friendly, helpful manner."""

# 시스템 프롬프트: 항공편 예약 전문가 - 항공편 검색, 예약, 항공사/노선/여행 정책 안내
FLIGHT_BOOKING_PROMPT = f"""You are a flight booking assistant. Help customers find flights, make reservations, and answer questions about airlines, routes, and travel policies. 
Provide clear information about flight availability, pricing, schedules, and booking procedures in a friendly, helpful manner."""

### 에이전트 도구 구현
이제 코디네이터 에이전트가 사용할 수 있는 도구로 전문화된 에이전트를 구현하겠습니다:

In [ ]:
@tool
def flight_booking_assistant(query: str) -> str:
    """
    Process and respond to flight booking queries.

    Args:
        query: A flight-related question about bookings, schedules, airlines, or travel policies

    Returns:
        Detailed flight information, booking options, or travel advice
    """
    try:
        flight_memory_hooks = ShortTermMemoryHook(client, memory_id)
        
        flight_agent = Agent(
            hooks=[flight_memory_hooks],
            model=MODEL_ID,
            system_prompt=FLIGHT_BOOKING_PROMPT,
            state={"actor_id": flight_actor_id, "session_id": session_id}
        )

        response = flight_agent(query)
        return str(response)
    except Exception as e:
        return f"항공편 예약 어시스턴트 오류: {str(e)}"

@tool
def hotel_booking_assistant(query: str) -> str:
    """
    Process and respond to hotel booking queries.

    Args:
        query: A hotel-related question about accommodations, amenities, or reservations

    Returns:
        Detailed hotel information, booking options, or accommodation advice
    """
    try:
        hotel_memory_hooks = ShortTermMemoryHook(client, memory_id)

        hotel_booking_agent = Agent(
            hooks=[hotel_memory_hooks],
            model=MODEL_ID,
            system_prompt=HOTEL_BOOKING_PROMPT,
            state={"actor_id": hotel_actor_id, "session_id": session_id}
        )
        
        response = hotel_booking_agent(query)
        return str(response)
    except Exception as e:
        return f"호텔 예약 어시스턴트 오류: {str(e)}"

### 코디네이터 에이전트 생성

마지막으로, 전문화된 도구들 사이를 조율하는 메인 여행 계획 에이전트를 만들겠습니다:

In [ ]:
# 시스템 프롬프트: 여행 코디네이터 - 항공편/호텔 전문 도구를 조율하여 종합적인 여행 계획 제공
TRAVEL_AGENT_SYSTEM_PROMPT = """
You are a comprehensive travel planning assistant that coordinates between specialized tools:
- For flight-related queries (bookings, schedules, airlines, routes) → Use the flight_booking_assistant tool
- For hotel-related queries (accommodations, amenities, reservations) → Use the hotel_booking_assistant tool
- For complete travel packages → Use both tools as needed to provide comprehensive information
- For general travel advice or simple travel questions → Answer directly

Each agent will have its own memory in case the user asks about historic data.
When handling complex travel requests, coordinate information from both tools to create a cohesive travel plan.
Provide clear organization when presenting information from multiple sources. \
Ask max two questions per turn. Keep the messages short, don't overwhelm the customer.
"""

In [ ]:
travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant]
)

#### 멀티 에이전트 시스템이 준비되었습니다!!

## 에이전트를 테스트해 봅시다.

여행 계획 시나리오로 멀티 에이전트 시스템을 테스트해 보겠습니다:

In [ ]:
# 사용자: 안녕하세요, LA에서 마드리드로 가는 여행을 예약하고 싶습니다. 7월 1일부터 8월 2일까지입니다.
response = travel_agent("Hello, I would like to book a trip from LA to Madrid. From July 1 to August 2.")

In [ ]:
# 사용자: 지금은 항공편에만 집중하고 싶습니다. 직항편, 중간 가격대, 도심, 수영장, 스탠다드 객실
response = travel_agent("I would only like to focus on the flight at the moment. direct flimid-range, city center, pool, standard room")

## 메모리 지속성 테스트

메모리 시스템이 올바르게 작동하는지 테스트하기 위해, 새로운 여행 에이전트 인스턴스를 생성하고 이전에 저장된 정보에 접근할 수 있는지 확인해 보겠습니다:

In [ ]:
# 여행 에이전트의 새 인스턴스 생성
new_travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant]
)

# 이전 대화에 대해 질문
# 사용자: 이전에 논의했던 항공편에 대해 알려줄 수 있나요?
new_travel_agent("Can you remind me about flights talked about before?")

## 요약

이 노트북에서 다음 내용을 시연했습니다:

1. 여러 에이전트를 위한 공유 메모리 리소스 생성 방법
2. 메모리 접근 권한이 있는 전문화된 에이전트를 도구로 구현하는 방법
3. 대화 컨텍스트를 유지하면서 여러 에이전트 간 조율하는 방법
4. 서로 다른 에이전트 인스턴스 간 메모리가 지속되는 방법

공유 메모리가 있는 이 멀티 에이전트 아키텍처는 전문화된 도메인을 처리하면서도 일관된 사용자 경험을 유지하는 복잡한 대화형 AI 시스템을 구축하기 위한 강력한 접근 방식을 제공합니다.

## 정리
이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제하겠습니다.

In [ ]:
#client.delete_memory_and_wait(
#        memory_id = memory_id,
#        max_wait = 300,
#        poll_interval =10
#)